# Examen Final - Diseno e Implementacion de un Sistema de Recuperacion de Informacion

**ICCD753 Recuperacion de Informacion - 2026-A**
**Prof. Ivan Carrera - EPN-FIS**
**Estudiante:** Daniel Flores

---

## Panorama general: que vamos a construir y para que sirve

Un sistema **RAG** (Retrieval-Augmented Generation, "generacion aumentada por recuperacion") es una
forma de hacer que un modelo de lenguaje (LLM) responda preguntas usando informacion externa
concreta, en vez de responder solo con lo que "memorizo" durante su entrenamiento. La idea central
es simple:

1. Guardamos un conjunto de documentos (en este caso, resumenes de articulos cientificos de arXiv).
2. Cuando llega una pregunta, buscamos los documentos mas parecidos semanticamente a la pregunta.
3. Le pasamos esos documentos al LLM como "contexto" y le pedimos que responda basandose en ellos.
4. Mostramos al usuario tanto la respuesta como los documentos usados, para que pueda verificarla.

Este patron se usa hoy en productos muy conocidos: los asistentes de busqueda de empresas (buscar
en manuales internos o tickets de soporte), los asistentes de codigo que citan la documentacion de
una libreria, o buscadores academicos que resumen papers relacionados con una pregunta. La ventaja
frente a "solo preguntarle al LLM" es que el sistema puede **citar sus fuentes** y puede **admitir
que no sabe** cuando el corpus no contiene la respuesta, en vez de inventar (alucinar) informacion.

En este notebook vamos a construir cada pieza de ese pipeline (preparacion del corpus, embeddings,
base vectorial, recuperacion + re-ranking, generacion, evidencias) y despues vamos a desplegarlo
como una aplicacion web de chat accesible desde un navegador.

**Dato curioso (algebra lineal):** en el fondo, "buscar el documento mas parecido a una pregunta"
es un problema de geometria: representamos cada texto como un vector en un espacio de cientos de
dimensiones, y "parecido" se mide con el **coseno del angulo** entre dos vectores. Dos vectores
apuntando casi en la misma direccion (angulo cercano a 0) tienen coseno cercano a 1 y se consideran
semanticamente similares, sin importar que tan "largos" sean los textos originales.


## Configuracion inicial

Antes de empezar con los requerimientos del examen, instalamos/importamos las librerias que vamos a
usar y cargamos la clave de la API de Groq desde un archivo `.env` (nunca directamente en el
codigo, para no exponerla si subimos el notebook a un repositorio publico).


In [1]:
# Instala las librerias que falten (no hace nada si ya estan instaladas)
# %pip install kagglehub pandas numpy fastembed faiss-cpu openai python-dotenv --quiet


In [2]:
import os
import json

import numpy as np
import pandas as pd
import faiss
from dotenv import load_dotenv
from openai import OpenAI
from fastembed import TextEmbedding

# load_dotenv() busca un archivo .env en esta carpeta y carga sus variables
# como si fueran variables de entorno del sistema.
load_dotenv()

# El cliente de Groq usa el mismo formato que la libreria de OpenAI, solo
# cambiamos la url base para que apunte a los servidores de Groq.
cliente = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)
MODELO_LLM = "llama-3.3-70b-versatile"

print("Cliente Groq listo, modelo:", MODELO_LLM)


/home/daniel/ir26a-danielFlores/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cliente Groq listo, modelo: llama-3.3-70b-versatile


## A. Preparacion del corpus

**Panorama:** la base de datos del examen es **arXiv Paper Abstracts**
(`spsayakpaul/arxiv-paper-abstracts` en Kaggle), una coleccion de titulos, resumenes (abstracts) y
categorias/temas (`terms`, por ejemplo `cs.LG` = machine learning o `cs.CV` = vision por
computador) de articulos cientificos publicados en arXiv.

El dataset completo tiene **51.774 filas**. Para un examen no necesitamos ni conviene usarlo
completo:

- El notebook tardaria mucho en calcular embeddings de 51 mil resumenes.
- La aplicacion web se despliega en una **funcion serverless de Vercel**, que tiene un limite de
  tamano por funcion. Si empaquetamos el corpus completo con sus embeddings, no cabe.

Por eso tomamos una **muestra representativa de 4000 articulos**. El proceso es:

1. Eliminar titulos duplicados (el dataset original trae varias filas casi identicas).
2. Revisar cuantos articulos mencionan ciertos temas de interes (para asegurarnos de que la
   muestra final sea util para las preguntas de ejemplo del examen: Graph Neural Networks,
   reinforcement learning, diffusion models, retrieval-augmented generation).
3. Incluir siempre los articulos que mencionan temas poco frecuentes (por ejemplo "diffusion
   model" solo aparece en 19 de 38972 articulos unicos), para que no desaparezcan por accidente
   al tomar una muestra aleatoria.
4. Completar el resto de la muestra con articulos aleatorios (semilla fija, para que el
   resultado sea reproducible) hasta llegar a 4000.

Esto es un **muestreo estratificado**: en vez de confiar ciegamente en el azar, garantizamos que
los temas raros pero relevantes para el examen sigan presentes en la muestra final.


In [3]:
# Carga del corpus tal como lo entrega Kaggle, usando kagglehub
# (mismo patron sugerido en el enunciado del examen)
import kagglehub

ruta_dataset = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")
ruta_csv = os.path.join(ruta_dataset, "arxiv_data.csv")

df = pd.read_csv(ruta_csv)
print("filas originales:", len(df))
df.head(3)


filas originales: 51774


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"


In [4]:
# Quitamos titulos duplicados: el dataset trae el mismo articulo repetido
# varias veces (probablemente por como fue recolectado desde arXiv).
df = df.drop_duplicates(subset=["titles"]).reset_index(drop=True)
print("filas sin duplicados de titulo:", len(df))


filas sin duplicados de titulo: 38972


In [5]:
# Construimos un texto en minusculas (titulo + resumen) para buscar palabras
# clave de los temas del examen y decidir que filas "forzamos" a incluir.
texto_busqueda = (df["titles"] + " " + df["summaries"]).str.lower()

es_diffusion = texto_busqueda.str.contains("diffusion model", regex=False)
es_rag = (
    texto_busqueda.str.contains("retrieval-augmented", regex=False)
    | texto_busqueda.str.contains("retrieval augmented", regex=False)
)

print("articulos que mencionan 'diffusion model':", es_diffusion.sum())
print("articulos que mencionan retrieval-augmented generation:", es_rag.sum())


articulos que mencionan 'diffusion model': 19
articulos que mencionan retrieval-augmented generation: 2


In [6]:
# Filas que garantizamos incluir (temas raros) + muestra aleatoria del resto,
# hasta completar 4000 articulos en total.
N_TOTAL = 4000
SEMILLA = 42

filas_forzadas = df[es_diffusion | es_rag]
resto = df[~(es_diffusion | es_rag)]

n_faltantes = N_TOTAL - len(filas_forzadas)
muestra_aleatoria = resto.sample(n=n_faltantes, random_state=SEMILLA)

corpus = pd.concat([filas_forzadas, muestra_aleatoria])
corpus = corpus.sample(frac=1, random_state=SEMILLA + 1).reset_index(drop=True)  # barajamos el orden

# Le damos un id corto a cada documento y renombramos columnas a espanol
corpus.insert(0, "doc_id", [f"arxiv_{i:05d}" for i in range(len(corpus))])
corpus = corpus.rename(columns={"titles": "titulo", "summaries": "abstract", "terms": "categorias"})

print("tamano final del corpus:", len(corpus))
corpus.to_csv("corpus_arxiv.csv", index=False)
corpus.head(3)


tamano final del corpus: 4000


,doc_id,titulo,abstract,categorias
0,arxiv_00000,KNH: Multi-View Modeling with K-Nearest Hyperp...,Graphs are one of the most efficacious structu...,"['cs.LG', 'cs.AI', 'cs.CY']"
1,arxiv_00001,Incremental Multi-Target Domain Adaptation for...,Recent advances in unsupervised domain adaptat...,['cs.CV']
2,arxiv_00002,Health Analytics: a systematic review of appro...,The paper presents a systematic review of stat...,['stat.ML']


## B. Representacion mediante embeddings

**Panorama:** un *embedding* es un vector de numeros que representa el significado de un texto.
Textos con significado parecido quedan como vectores cercanos en ese espacio, aunque usen palabras
distintas ("carro" y "automovil" quedarian cerca; "carro" y "manzana" quedarian lejos). Este es el
mismo principio que usamos en el ejercicio de bases de datos vectoriales y en el taller de CLIP,
pero aqui elegimos un modelo distinto por una razon practica.

**Por que un modelo "liviano" (fastembed / BAAI/bge-small-en-v1.5) y no uno mas pesado como
`sentence-transformers` con PyTorch:** la version de produccion de este sistema corre dentro de una
**funcion serverless en Vercel**, que tiene un limite estricto de tamano (unos 250 MB por funcion).
`torch` + `sentence-transformers` facilmente ocupan mas de 1 GB, asi que no caben. `fastembed` usa
**ONNX Runtime** en vez de PyTorch: el mismo modelo, convertido a un formato mas compacto y rapido
para hacer solo inferencia (no entrenamiento), lo que pesa una fraccion de eso. Usamos el mismo
modelo tanto en este notebook como en la funcion desplegada, para que las preguntas del usuario y
los documentos del corpus queden representados en el **mismo espacio vectorial** (si usaramos un
modelo distinto en cada lado, las distancias dejarian de tener sentido).

Tambien recortamos cada resumen a los primeros 600 caracteres antes de calcular su embedding (no
para mostrarlo al usuario, solo para el calculo del vector). Los primeros 600 caracteres de un
abstract cientifico ya contienen la idea principal del articulo, y calcular el embedding de un
texto mas corto es varias veces mas rapido (el costo de estos modelos crece de forma no lineal con
la longitud del texto).


In [7]:
# Cargamos el modelo de embeddings liviano (se descarga una sola vez y
# queda en cache local).
modelo_embed = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Texto que se usa para calcular el embedding: titulo + primeros 600
# caracteres del abstract. El abstract completo se guarda aparte para
# mostrarlo como evidencia.
textos_para_embeber = (
    corpus["titulo"] + ". " + corpus["abstract"].str.slice(0, 600)
).tolist()

print("cantidad de documentos a embeber:", len(textos_para_embeber))


cantidad de documentos a embeber: 4000


In [8]:
# Calculamos un vector por documento. Esto puede tardar un par de minutos
# porque son 4000 documentos.
embeddings_corpus = list(modelo_embed.embed(textos_para_embeber, batch_size=64))
embeddings_corpus = np.array(embeddings_corpus, dtype="float32")

print("forma de la matriz de embeddings:", embeddings_corpus.shape)
print("norma promedio de cada vector (~1.0 si esta normalizado):", np.linalg.norm(embeddings_corpus, axis=1).mean())

np.save("corpus_embeddings.npy", embeddings_corpus)


forma de la matriz de embeddings: (4000, 384)
norma promedio de cada vector (~1.0 si esta normalizado): 1.0


## C. Almacenamiento y busqueda vectorial

**Panorama:** ya tenemos 4000 vectores de 384 numeros cada uno. Para buscar "los documentos mas
parecidos a esta pregunta" necesitamos comparar el vector de la pregunta contra los 4000 vectores
del corpus y quedarnos con los mas cercanos. Hacer esto "a mano" con un ciclo en Python seria lento;
una **base de datos vectorial** hace esta comparacion de forma optimizada (usando operaciones
matriciales y, en datasets mas grandes, estructuras de indice que evitan comparar contra todos los
vectores).

Usamos **FAISS** (Facebook AI Similarity Search), la misma libreria que exploramos en el ejercicio
de bases de datos vectoriales. En concreto usamos `IndexFlatIP`: un indice que hace **busqueda
exacta** por producto punto (Inner Product). Como nuestros vectores estan normalizados (norma 1),
el producto punto entre dos vectores es exactamente el **coseno del angulo** entre ellos, que es la
metrica de similitud semantica estandar para embeddings de texto.

**Dato curioso (complejidad algoritmica):** `IndexFlatIP` es basicamente una busqueda lineal, O(n)
por consulta: compara contra los n vectores del corpus. Con 4000 documentos esto toma milisegundos,
pero si el corpus tuviera millones de documentos, se usarian indices aproximados (como HNSW, un
grafo de vecinos cercanos) que buscan en tiempo sub-lineal a cambio de perder un poco de precision.
Para el tamano de este examen, la busqueda exacta es la opcion correcta: es simple y ya es rapida.


In [9]:
# Creamos un indice FAISS para busqueda por producto punto (equivalente a
# similitud coseno porque los vectores estan normalizados).
dimension = embeddings_corpus.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)
indice_faiss.add(embeddings_corpus)

print("dimension de cada vector:", dimension)
print("vectores almacenados en el indice:", indice_faiss.ntotal)


dimension de cada vector: 384
vectores almacenados en el indice: 4000


In [10]:
# Prueba rapida: buscamos el propio documento 0 dentro del indice.
# El resultado con mayor similitud deberia ser el mismo documento (similitud ~1.0).
similitudes_prueba, posiciones_prueba = indice_faiss.search(embeddings_corpus[0:1], 3)
print("similitudes:", similitudes_prueba[0])
print("titulos mas cercanos al documento 0:")
for pos in posiciones_prueba[0]:
    print(" -", corpus.iloc[pos]["titulo"])


similitudes: [1.        0.7771556 0.7766527]
titulos mas cercanos al documento 0:
 - KNH: Multi-View Modeling with K-Nearest Hyperplanes Graph for Misinformation Detection
 - Fast Graph Attention Networks Using Effective Resistance Based Graph Sparsification
 - Visualizing Graph Neural Networks with CorGIE: Corresponding a Graph to Its Embedding


## D. Recuperacion

**Panorama:** la recuperacion tiene dos pasos en este sistema:

1. **Busqueda semantica (embeddings):** convertimos la pregunta del usuario en un vector con el
   mismo modelo usado para el corpus, y le pedimos a FAISS los `k` documentos mas parecidos
   (por ejemplo, los 15 mejores). Esto es rapido pero no siempre perfecto: la similitud de
   embeddings puede confundir temas relacionados aunque no sean exactamente lo que se pregunto.

2. **Re-ranking con LLM:** de esos 15 candidatos, le pedimos al LLM (Groq) que lea la pregunta y
   los titulos/resumenes de los candidatos, y elija los 5 realmente mas relevantes, ordenados. Un
   LLM puede razonar sobre el *significado* de la pregunta de una forma mas fina que una simple
   distancia vectorial, asi que el re-ranking normalmente mejora la calidad del top final. Esta
   es la parte "moderna" del enfoque que pide el examen: no nos quedamos solo con el resultado
   crudo de la busqueda vectorial, sino que lo refinamos con el propio LLM antes de generar la
   respuesta.

Le pedimos al LLM que responda en **JSON estricto** (`{"ranked_ids": [...]}`) para poder leer su
respuesta de forma confiable con codigo, en vez de tener que interpretar texto libre.


In [11]:
def recuperar(pregunta, k=15):
    """Busca los k documentos mas parecidos a la pregunta usando el indice FAISS."""
    vector_pregunta = np.array(list(modelo_embed.embed([pregunta])), dtype="float32")
    similitudes, posiciones = indice_faiss.search(vector_pregunta, k)

    candidatos = corpus.iloc[posiciones[0]].copy()
    candidatos["similitud"] = similitudes[0]
    return candidatos.reset_index(drop=True)


In [12]:
# Probamos solo la busqueda semantica (sin re-ranking todavia) con una
# pregunta de ejemplo del enunciado del examen.
pregunta_ejemplo = "What are the main applications of Graph Neural Networks?"
candidatos_ejemplo = recuperar(pregunta_ejemplo, k=15)

for fila in candidatos_ejemplo.itertuples():
    print(f"{fila.similitud:.3f}  {fila.titulo}")


0.808  Should Graph Neural Networks Use Features, Edges, Or Both?
0.793  Graph Neural Networks for Graph Drawing
0.785  Self-Enhanced GNN: Improving Graph Neural Networks Using Model Outputs
0.782  Strategies for Pre-training Graph Neural Networks
0.781  Rethinking Graph Neural Architecture Search from Message-passing
0.778  Increase and Conquer: Training Graph Neural Networks on Growing Graphs
0.778  GraphMDN: Leveraging graph structure and deep learning to solve inverse problems
0.777  Graph Self Supervised Learning: the BT, the HSIC, and the VICReg
0.774  EqGNN: Equalized Node Opportunity in Graphs
0.774  Graph Random Neural Network for Semi-Supervised Learning on Graphs
0.770  Policy-GNN: Aggregation Optimization for Graph Neural Networks
0.769  Learning Graph Normalization for Graph Neural Networks
0.768  Graph Neural Networks for Decentralized Controllers
0.766  Visualizing Graph Neural Networks with CorGIE: Corresponding a Graph to Its Embedding
0.765  Graph Neural Network for T

In [13]:
def rerankear(pregunta, candidatos, top_n=5):
    """Le pide al LLM que elija y ordene los top_n candidatos mas relevantes."""
    lista_candidatos = "\n".join(
        f"[{fila.doc_id}] {fila.titulo}: {fila.abstract[:300]}"
        for fila in candidatos.itertuples()
    )

    prompt = f"""Eres un sistema de re-ranking para busqueda academica.
Pregunta del usuario: "{pregunta}"

Candidatos (id, titulo, resumen recortado):
{lista_candidatos}

Elige los {top_n} candidatos MAS relevantes para responder la pregunta, ordenados del mas al menos relevante.
Responde solo JSON: {{"ranked_ids": ["id1", "id2", ...]}}"""

    respuesta = cliente.chat.completions.create(
        model=MODELO_LLM,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )

    try:
        ids_ordenados = json.loads(respuesta.choices[0].message.content)["ranked_ids"]
    except Exception:
        # Si el LLM no devuelve un JSON valido, usamos el orden original de FAISS.
        ids_ordenados = candidatos["doc_id"].tolist()[:top_n]

    candidatos_por_id = candidatos.set_index("doc_id")
    ids_validos = [i for i in ids_ordenados if i in candidatos_por_id.index]
    if not ids_validos:
        ids_validos = candidatos["doc_id"].tolist()[:top_n]

    return candidatos_por_id.loc[ids_validos[:top_n]].reset_index()


In [14]:
# Aplicamos el re-ranking sobre los candidatos anteriores.
evidencia_ejemplo = rerankear(pregunta_ejemplo, candidatos_ejemplo, top_n=5)
evidencia_ejemplo[["doc_id", "titulo", "similitud"]]


,doc_id,titulo,similitud
0,arxiv_02711,Graph Neural Network for Traffic Forecasting: ...,0.765279
1,arxiv_02023,Graph Neural Networks for Graph Drawing,0.793482
2,arxiv_02052,Graph Neural Networks for Decentralized Contro...,0.767812
3,arxiv_00756,Learning Graph Normalization for Graph Neural ...,0.768992
4,arxiv_03027,Policy-GNN: Aggregation Optimization for Graph...,0.770271


## E. Generacion aumentada por recuperacion

**Panorama:** con los documentos ya seleccionados (top 5 tras el re-ranking), construimos un
"contexto" con sus titulos y resumenes completos, y se lo damos al LLM junto con instrucciones
claras:

- Responder **solo** con base en ese contexto (no usar conocimiento general del modelo).
- Citar los documentos usados con su id, por ejemplo `[Documento arxiv_01234]`.
- Si el contexto no alcanza para responder con confianza, **decirlo explicitamente** en vez de
  inventar una respuesta (esto es el requerimiento del examen sobre reconocer cuando el corpus no
  tiene informacion suficiente).

Esto se implementa con dos mensajes: un mensaje de **sistema** (las reglas de comportamiento) y un
mensaje de **usuario** (el contexto + la pregunta real).


In [15]:
def generar_respuesta(pregunta, evidencia):
    """Genera la respuesta final del LLM usando los documentos de 'evidencia' como contexto."""
    contexto = "\n\n".join(
        f"[Documento {fila.doc_id}] {fila.titulo}\n{fila.abstract}"
        for fila in evidencia.itertuples()
    )

    prompt_sistema = (
        "Eres un asistente que responde preguntas sobre articulos cientificos de arXiv. "
        "Responde UNICAMENTE con base en los documentos de contexto que se te dan. "
        "Cita los documentos relevantes usando su [Documento id]. "
        "Si el contexto no contiene informacion suficiente para responder con confianza, "
        "dilo explicitamente en vez de inventar una respuesta."
    )
    prompt_usuario = f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"

    respuesta = cliente.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario},
        ],
        temperature=0.2,
    )
    return respuesta.choices[0].message.content


In [16]:
respuesta_ejemplo = generar_respuesta(pregunta_ejemplo, evidencia_ejemplo)
print(respuesta_ejemplo)


Según los documentos proporcionados, las principales aplicaciones de las Redes Neuronales de Grafos (Graph Neural Networks, GNN) incluyen:

1. **Predicción de tráfico**: En [Documento arxiv_02711], se explora el uso de GNN para la predicción de tráfico, incluyendo la predicción de flujo y velocidad de tráfico en carreteras y la predicción de flujo de pasajeros en sistemas de transporte público.
2. **Diseño de grafos**: En [Documento arxiv_02023], se propone un marco para el desarrollo de "Graph Neural Drawers" (GND), que utilizan GNN para construir mapas eficientes y complejos.
3. **Controladores descentralizados**: En [Documento arxiv_02052], se propone un marco para el aprendizaje de controladores descentralizados utilizando GNN, con aplicaciones en sistemas de agentes autónomos, como la robótica y las redes inteligentes.
4. **Procesamiento de datos de grafos**: En [Documento arxiv_00756] y [Documento arxiv_03027], se explora el uso de GNN para el procesamiento de datos de grafos, in

## F. Presentacion de evidencias

**Panorama:** un LLM puede sonar convincente incluso cuando se equivoca. Por eso el examen pide
explicitamente que el sistema **muestre los fragmentos recuperados** y no solo la respuesta del
LLM: asi el usuario puede verificar por si mismo si la respuesta realmente se apoya en los
documentos, o si el LLM se desvio del contexto.

Juntamos todo el pipeline (`recuperar` -> `rerankear` -> `generar_respuesta`) en una sola funcion
`responder_con_evidencia`, que devuelve la respuesta **y** la lista de documentos usados como
evidencia (con su similitud y un fragmento del abstract). Esta es la misma funcion, en esencia, que
usa la aplicacion web desplegada.


In [17]:
def responder_con_evidencia(pregunta, k=15, top_n=5):
    candidatos = recuperar(pregunta, k=k)
    evidencia = rerankear(pregunta, candidatos, top_n=top_n)
    respuesta = generar_respuesta(pregunta, evidencia)
    return respuesta, evidencia


def mostrar_resultado(pregunta, respuesta, evidencia):
    print("PREGUNTA:", pregunta)
    print()
    print("EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):")
    for fila in evidencia.itertuples():
        print(f"  [{fila.doc_id}] similitud={fila.similitud:.3f}  {fila.titulo}")
        print(f"      {fila.abstract[:180]}...")
    print()
    print("RESPUESTA DEL LLM:")
    print(respuesta)
    print("=" * 100)


In [18]:
respuesta_demo, evidencia_demo = responder_con_evidencia(pregunta_ejemplo)
mostrar_resultado(pregunta_ejemplo, respuesta_demo, evidencia_demo)


PREGUNTA: What are the main applications of Graph Neural Networks?

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_02711] similitud=0.765  Graph Neural Network for Traffic Forecasting: A Survey
      Traffic forecasting is important for the success of intelligent
transportation systems. Deep learning models, including convolution neural
networks and recurrent neural networks, h...
  [arxiv_02023] similitud=0.793  Graph Neural Networks for Graph Drawing
      Graph Drawing techniques have been developed in the last few years with the
purpose of producing aesthetically pleasing node-link layouts. Recently, the
employment of differentiabl...
  [arxiv_02052] similitud=0.768  Graph Neural Networks for Decentralized Controllers
      Dynamical systems comprised of autonomous agents arise in many relevant
problems such as multi-agent robotics, smart grids, or smart cities.
Controlling these systems is of paramou...
  [arxiv_03027] similitud=0.770  Policy-GNN: Aggregation 

## G. Interfaz web conversacional

**Panorama:** el examen pide una interfaz web tipo chat, no una interfaz de notebook. Construimos
una pagina simple (`index.html`, HTML + CSS + JavaScript en un solo archivo, sin frameworks) que:

- Tiene un cuadro de texto para escribir preguntas en lenguaje natural.
- Muestra la respuesta del LLM en una burbuja de chat.
- Muestra la evidencia (documentos recuperados) en un panel desplegable debajo de cada respuesta.
- Permite hacer una nueva pregunta sin recargar la pagina (usa `fetch` para llamar a la API cada
  vez, de forma independiente).

**No implementamos memoria conversacional** porque el examen indica explicitamente que no es
obligatoria: cada pregunta se procesa de forma independiente, sin recordar preguntas anteriores.

La logica de backend (recuperar + rerankear + generar) vive en una **funcion serverless de Python**
(`api/chat.py`) que Vercel expone como el endpoint `/api/chat`. Es practicamente el mismo codigo que
ya escribimos arriba en el notebook, empaquetado como funcion HTTP. La siguiente celda escribe ese
archivo a disco (es el codigo real que se despliega, no un resumen).


In [19]:
%%writefile webapp/api/chat.py
import os
import json
from http.server import BaseHTTPRequestHandler

import numpy as np
from openai import OpenAI
from fastembed import TextEmbedding

DATA_DIR = os.path.join(os.path.dirname(__file__), "data")

with open(os.path.join(DATA_DIR, "corpus.json"), encoding="utf-8") as f:
    CORPUS = json.load(f)

EMBEDDINGS = np.load(os.path.join(DATA_DIR, "embeddings.npy"))

EMBED_MODEL = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)
MODELO_LLM = "llama-3.3-70b-versatile"


def recuperar(pregunta, k=15):
    vector = np.array(list(EMBED_MODEL.embed([pregunta])), dtype="float32")[0]
    similitudes = EMBEDDINGS @ vector
    mejores_posiciones = np.argsort(-similitudes)[:k]
    return [
        {**CORPUS[i], "similitud": float(similitudes[i])}
        for i in mejores_posiciones
    ]


def rerankear(pregunta, candidatos, top_n=5):
    lista = "\n".join(
        f"[{c['doc_id']}] {c['titulo']}: {c['abstract'][:300]}" for c in candidatos
    )
    prompt = f"""Eres un sistema de re-ranking para busqueda academica.
Pregunta del usuario: "{pregunta}"

Candidatos (id, titulo, resumen recortado):
{lista}

Elige los {top_n} candidatos MAS relevantes para responder la pregunta, ordenados del mas al menos relevante.
Responde solo JSON: {{"ranked_ids": ["id1", "id2", ...]}}"""

    resp = client.chat.completions.create(
        model=MODELO_LLM,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"},
    )
    try:
        ids_ordenados = json.loads(resp.choices[0].message.content)["ranked_ids"]
    except Exception:
        ids_ordenados = [c["doc_id"] for c in candidatos[:top_n]]

    por_id = {c["doc_id"]: c for c in candidatos}
    seleccion = [por_id[i] for i in ids_ordenados if i in por_id]
    if not seleccion:
        seleccion = candidatos[:top_n]
    return seleccion[:top_n]


def generar_respuesta(pregunta, evidencia):
    contexto = "\n\n".join(
        f"[Documento {c['doc_id']}] {c['titulo']}\n{c['abstract']}" for c in evidencia
    )
    prompt_sistema = (
        "Eres un asistente que responde preguntas sobre articulos cientificos de arXiv. "
        "Responde UNICAMENTE con base en los documentos de contexto que se te dan. "
        "Cita los documentos relevantes usando su [Documento id]. "
        "Si el contexto no contiene informacion suficiente para responder con confianza, "
        "dilo explicitamente en vez de inventar una respuesta."
    )
    prompt_usuario = f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"

    resp = client.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario},
        ],
        temperature=0.2,
    )
    return resp.choices[0].message.content


class handler(BaseHTTPRequestHandler):
    def do_POST(self):
        try:
            largo = int(self.headers.get("Content-Length", 0))
            cuerpo = json.loads(self.rfile.read(largo) or b"{}")
            pregunta = (cuerpo.get("pregunta") or "").strip()
            if not pregunta:
                self._responder(400, {"error": "Falta el campo 'pregunta'."})
                return

            candidatos = recuperar(pregunta, k=15)
            evidencia = rerankear(pregunta, candidatos, top_n=5)
            respuesta = generar_respuesta(pregunta, evidencia)

            self._responder(200, {
                "respuesta": respuesta,
                "evidencia": [
                    {
                        "doc_id": c["doc_id"],
                        "titulo": c["titulo"],
                        "abstract": c["abstract"][:500],
                        "categorias": c.get("categorias", ""),
                        "similitud": round(c["similitud"], 4),
                    }
                    for c in evidencia
                ],
            })
        except Exception as e:
            self._responder(500, {"error": str(e)})

    def _responder(self, codigo, cuerpo):
        self.send_response(codigo)
        self.send_header("Content-Type", "application/json")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(json.dumps(cuerpo).encode("utf-8"))

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.end_headers()


Overwriting webapp/api/chat.py


La interfaz (`webapp/index.html`) es una pagina de chat sencilla: mantiene una lista de
mensajes en el DOM, y por cada pregunta hace un `fetch` a `/api/chat`, muestra la respuesta y agrega
un bloque `<details>` (desplegable) con los documentos de evidencia, su similitud y un fragmento del
abstract. El codigo completo esta en `webapp/index.html` dentro de este mismo folder del examen.


## H. Despliegue en la nube

**Panorama:** desplegamos la aplicacion en **Vercel**. El proyecto tiene dos partes:

- `webapp/index.html`: archivo estatico, servido directamente por la red de Vercel.
- `webapp/api/chat.py`: funcion serverless en Python que Vercel detecta automaticamente por estar
  dentro de la carpeta `api/` y expone como el endpoint `POST /api/chat`.

La clave de la API de Groq (`GROQ_API_KEY`) se configura como **variable de entorno del proyecto en
Vercel** (Project Settings -> Environment Variables), nunca en el codigo fuente ni en el
repositorio de Git. El archivo `.env` de este notebook esta ademas excluido del repositorio
mediante `.gitignore` (`.env` y `**/.env`).

**URL desplegada:** `PENDIENTE_URL_VERCEL`


In [20]:
# Verificacion de que la aplicacion desplegada realmente responde.
import urllib.request

URL_DESPLEGADA = "PENDIENTE_URL_VERCEL"

try:
    with urllib.request.urlopen(URL_DESPLEGADA, timeout=10) as resp:
        print("Codigo de respuesta:", resp.status)
        print("La aplicacion esta activa en:", URL_DESPLEGADA)
except Exception as e:
    print("No se pudo verificar automaticamente (revisar manualmente en el navegador):", e)


No se pudo verificar automaticamente (revisar manualmente en el navegador): unknown url type: 'PENDIENTE_URL_VERCEL'


## I. Evaluacion del sistema y de la generacion

**Panorama:** el examen pide explicitamente un **juicio subjetivo en lenguaje natural**, no
metricas estandar de Recuperacion de Informacion como precision, recall o `qrels` (que requieren
un conjunto de relevancia etiquetado a mano para cada consulta, algo que no tenemos aqui). En vez
de eso, para cada pregunta miramos "a ojo" la respuesta y la evidencia, y escribimos por que nos
parece correcta o no, evaluando:

- **Correccion** de la respuesta.
- **Relevancia** respecto a la pregunta.
- **Fidelidad** respecto a las evidencias recuperadas (¿la respuesta dice solo lo que dicen los
  documentos, o se inventa algo?).
- **Integracion de varios documentos** (¿combina informacion de mas de un paper, o solo repite
  uno?).
- **Reconocimiento de informacion insuficiente** (¿admite cuando el corpus no alcanza para
  responder, en vez de inventar?).

Probamos las 4 preguntas de ejemplo del enunciado del examen, mas una quinta pregunta sobre un
tema que sabemos que **no** esta en el corpus (criptografia cuantica), para verificar
especificamente el ultimo punto.


In [21]:
preguntas_evaluacion = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
    "How can quantum cryptography be used to secure communications?",
]

resultados_evaluacion = []
for pregunta in preguntas_evaluacion:
    respuesta, evidencia = responder_con_evidencia(pregunta)
    mostrar_resultado(pregunta, respuesta, evidencia)
    resultados_evaluacion.append((pregunta, respuesta, evidencia))


PREGUNTA: What are the main applications of Graph Neural Networks?

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_02711] similitud=0.765  Graph Neural Network for Traffic Forecasting: A Survey
      Traffic forecasting is important for the success of intelligent
transportation systems. Deep learning models, including convolution neural
networks and recurrent neural networks, h...
  [arxiv_02023] similitud=0.793  Graph Neural Networks for Graph Drawing
      Graph Drawing techniques have been developed in the last few years with the
purpose of producing aesthetically pleasing node-link layouts. Recently, the
employment of differentiabl...
  [arxiv_02052] similitud=0.768  Graph Neural Networks for Decentralized Controllers
      Dynamical systems comprised of autonomous agents arise in many relevant
problems such as multi-agent robotics, smart grids, or smart cities.
Controlling these systems is of paramou...
  [arxiv_00756] similitud=0.769  Learning Graph Normaliza

PREGUNTA: How is reinforcement learning used in robotics?

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_03910] similitud=0.874  Using Deep Reinforcement Learning for the Continuous Control of Robotic Arms
      Deep reinforcement learning enables algorithms to learn complex behavior,
deal with continuous action spaces and find good strategies in environments
with high dimensional state sp...
  [arxiv_00096] similitud=0.849  Meta Reinforcement Learning for Sim-to-real Domain Adaptation
      Modern reinforcement learning methods suffer from low sample efficiency and
unsafe exploration, making it infeasible to train robotic policies entirely on
real hardware. In this wo...
  [arxiv_01515] similitud=0.844  Safety Augmented Value Estimation from Demonstrations (SAVED): Safe Deep Model-Based RL for Sparse Cost Robotic Tasks
      Reinforcement learning (RL) for robotics is challenging due to the difficulty
in hand-engineering a dense cost function, which can lead to u

PREGUNTA: Recent advances in diffusion models for image generation.

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_00050] similitud=0.877  Cascaded Diffusion Models for High Fidelity Image Generation
      We show that cascaded diffusion models are capable of generating high
fidelity images on the class-conditional ImageNet generation challenge, without
any assistance from auxiliary ...
  [arxiv_02263] similitud=0.813  Denoising Diffusion Implicit Models
      Denoising diffusion probabilistic models (DDPMs) have achieved high quality
image generation without adversarial training, yet they require simulating a
Markov chain for many steps...
  [arxiv_02400] similitud=0.813  3D Shape Generation and Completion through Point-Voxel Diffusion
      We propose a novel approach for probabilistic generative modeling of 3D
shapes. Unlike most existing models that learn to deterministically translate a
latent vector to a shape, ou...
  [arxiv_03274] similitud=0.810  Noise Es

PREGUNTA: Techniques for improving retrieval-augmented generation systems.

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_01114] similitud=0.746  Retrieval-Augmented Generation for Code Summarization via Hybrid GNN
      Source code summarization aims to generate natural language summaries from
structured code snippets for better understanding code functionalities.
However, automatic code summariza...
  [arxiv_00447] similitud=0.786  Hybrid Retrieval-Generation Reinforced Agent for Medical Image Report Generation
      Generating long and coherent reports to describe medical images poses
challenges to bridging visual patterns with informative human linguistic
descriptions. We propose a novel Hybr...
  [arxiv_01866] similitud=0.752  A Retrieve-and-Edit Framework for Predicting Structured Outputs
      For the task of generating complex outputs such as source code, editing
existing outputs can be easier than generating complex outputs from scratch.
With this motivat

PREGUNTA: How can quantum cryptography be used to secure communications?

EVIDENCIA RECUPERADA (antes de leer la respuesta del LLM):
  [arxiv_00477] similitud=0.613  Encryption Inspired Adversarial Defense for Visual Classification
      Conventional adversarial defenses reduce classification accuracy whether or
not a model is under attacks. Moreover, most of image processing based defenses
are defeated due to the ...
  [arxiv_01179] similitud=0.609  Block-wise Image Transformation with Secret Key for Adversarially Robust Defense
      In this paper, we propose a novel defensive transformation that enables us to
maintain a high classification accuracy under the use of both clean images and
adversarial examples fo...
  [arxiv_00710] similitud=0.608  Integer-arithmetic-only Certified Robustness for Quantized Neural Networks
      Adversarial data examples have drawn significant attention from the machine
learning and security communities. A line of work on tackling adversarial
examples i

### Juicio subjetivo por pregunta

*(El texto de abajo se escribio revisando la ejecucion real de la celda anterior, con las
respuestas y evidencias que efectivamente devolvio el sistema. Si se vuelve a ejecutar el notebook
el LLM puede redactar la respuesta con otras palabras, pero el patron general deberia mantenerse
porque depende sobre todo de que tan bien cubre el corpus a cada tema.)*

**1. "What are the main applications of Graph Neural Networks?"**
Correcta y relevante. La evidencia recuperada (similitud entre 0.77 y 0.81) son papers realmente
sobre GNNs: uso de features vs. edges, dibujo de grafos, prediccion de trafico, problemas inversos
y optimizacion de agregacion. La respuesta lista aplicaciones que efectivamente vienen de esos
documentos (clasificacion de grafos, prediccion de trafico, dibujo de grafos, problemas inversos) y
cita el `[arxiv_id]` de cada una, por lo que es fiel a la evidencia. Tambien integra varios
documentos distintos en vez de parafrasear uno solo, y cierra reconociendo que el contexto no es
exhaustivo, lo cual es una forma honesta de manejar la incertidumbre sin inventar aplicaciones que
no esten en los documentos.

**2. "How is reinforcement learning used in robotics?"**
Es la respuesta mas solida de las cinco. La similitud de los 5 documentos recuperados es alta
(0.82-0.87) y todos son directamente sobre RL aplicado a brazos roboticos, adaptacion sim-to-real,
manipulacion y control seguro. La respuesta integra los 5 documentos de forma coherente (control
continuo, adaptacion de dominio, RL basado en modelos, control de fuerza en manipulacion) y cita
cada afirmacion con su id, en vez de mezclar todo sin distincion. Es correcta, relevante y fiel a
la evidencia.

**3. "Recent advances in diffusion models for image generation."**
Contra lo que se esperaria por lo poco frecuente que es el tema en el dataset (solo 19 de casi 39
mil articulos unicos mencionan "diffusion model"), la respuesta salio sorprendentemente buena: la
similitud de los documentos recuperados es la mas alta de las cinco preguntas (0.80-0.88), porque
al forzar la inclusion de esos 19 articulos en la muestra nos aseguramos de que los pocos papers
relevantes que existen (modelos en cascada, DDIMs, estimacion de ruido, generacion 3D con difusion)
quedaran disponibles para el sistema. La respuesta integra varios de esos papers correctamente y es
fiel a lo que dicen. Esto muestra que la limitante real no era el motor de busqueda sino la
cobertura del corpus, y que una muestra bien construida puede compensar parcialmente un tema poco
frecuente.

**4. "Techniques for improving retrieval-augmented generation systems."**
Es una respuesta parcialmente correcta pero mas fronteriza. El termino "retrieval-augmented
generation" como tal casi no aparece en el corpus (2 menciones en casi 39 mil articulos unicos,
porque el dataset es de 2021 y el termino recien se popularizo despues), asi que la similitud de
los documentos recuperados es mas baja (0.71-0.79) y varios son sobre temas relacionados pero no
identicos (generacion de reportes medicos, prediccion de salidas estructuradas, atencion
jerarquica). La respuesta es fiel a esos documentos (no inventa tecnicas que no esten ahi) y los
integra razonablemente, pero el resultado es mas una respuesta sobre "recuperacion + generacion en
general" que sobre RAG especificamente. Es un buen ejemplo de una respuesta honesta con evidencia
imperfecta, aunque no llega a decir explicitamente que la cobertura es limitada.

**5. "How can quantum cryptography be used to secure communications?"**
Este es el caso mas claro de los cinco. El tema no aparece en el corpus (0 menciones de "quantum
crypt" en el dataset completo) y la similitud de los documentos recuperados es notablemente mas
baja que en las otras preguntas (0.61-0.64), la senal de que no hay nada realmente relevante. El
sistema responde explicitamente: *"No hay informacion suficiente en el contexto proporcionado para
responder con confianza..."*, describe de que tratan los documentos recuperados (privacidad en
analisis de datos, RL con privacidad diferencial, teoria cuantica de representaciones) y aclara que
ninguno responde la pregunta. Esto es exactamente el comportamiento que pide el examen: reconocer
los limites del corpus en vez de usar el conocimiento general del LLM sobre criptografia cuantica.

**Patron general:** las preguntas 1, 2 y 3 (con evidencia de similitud alta, 0.77-0.88) generan
respuestas solidas y bien fundamentadas; la pregunta 4 (similitud media, 0.71-0.79) genera una
respuesta correcta pero mas generica; la pregunta 5 (similitud baja, 0.61-0.64) hace que el sistema
reconozca correctamente que no puede responder. La calidad de la respuesta esta atada a que tan
buena es la evidencia recuperada, no a lo que el LLM "cree saber" por su cuenta, que es justamente
el comportamiento que se busca en un sistema RAG bien construido.


## Resumen final

| Requerimiento | Donde se cumple |
|---|---|
| A. Preparacion del corpus | Seccion A |
| B. Representacion mediante embeddings | Seccion B |
| C. Almacenamiento y busqueda vectorial | Seccion C (FAISS) |
| D. Recuperacion (+ re-ranking) | Seccion D |
| E. Generacion aumentada por recuperacion | Seccion E |
| F. Presentacion de evidencias | Seccion F |
| G. Interfaz web conversacional | Seccion G + `webapp/index.html` |
| H. Despliegue en la nube | Seccion H |
| I. Evaluacion (juicio subjetivo) | Seccion I |

**URL de la aplicacion desplegada:** `PENDIENTE_URL_VERCEL`
